In [1]:
import requests
import json
from pathlib import Path
from datetime import datetime, timezone


BASE_URL = "https://dummyjson.com/products"

LIMIT = 30

OUTPUT_DIR = Path("data/raw/products")


def fetch_products():
    all_products = []
    skip = 0

    while True:
        url = f"{BASE_URL}?limit={LIMIT}&skip={skip}"

        print(f"Fetching: skip={skip}, limit={LIMIT}")

        response = requests.get(url, timeout=30)
        response.raise_for_status()

        data = response.json()

        products = data["products"]
        total = data["total"]

        all_products.extend(products)

        print(f"Received {len(products)} products")

        skip += LIMIT

        if skip >= total:
            break

    return all_products


def save_raw_data(products):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    ingestion_time = datetime.now(timezone.utc)

    output = {
        "ingestion_timestamp": ingestion_time.isoformat(),
        "source": "dummyjson",
        "endpoint": BASE_URL,
        "record_count": len(products),
        "products": products,
    }

    file_name = (
        f"products_{ingestion_time.strftime('%Y%m%d_%H%M%S')}.json"
    )

    file_path = OUTPUT_DIR / file_name

    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2)

    print(f"Saved: {file_path}")


def main():
    products = fetch_products()
    save_raw_data(products)

    print(f"Total products: {len(products)}")


if __name__ == "__main__":
    main()

Fetching: skip=0, limit=30
Received 30 products
Fetching: skip=30, limit=30
Received 30 products
Fetching: skip=60, limit=30
Received 30 products
Fetching: skip=90, limit=30
Received 30 products
Fetching: skip=120, limit=30
Received 30 products
Fetching: skip=150, limit=30
Received 30 products
Fetching: skip=180, limit=30
Received 14 products
Saved: data\raw\products\products_20260907_192656.json
Total products: 194


In [10]:
import requests
import time


class APIClient:

    def __init__(self, base_url, max_retries=3, retry_delay=2):
        self.base_url = base_url.rstrip("/")
        self.max_retries = max_retries
        self.retry_delay = retry_delay

    def get_all(self, endpoint, data_key, limit=30):

        all_records = []
        skip = 0

        while True:

            url = f"{self.base_url}{endpoint}"

            for attempt in range(self.max_retries + 1):

                response = requests.get(
                    url,
                    params={
                        "limit": limit,
                        "skip": skip
                    },
                    timeout=30
                )

                if response.status_code == 429:

                    if attempt == self.max_retries:
                        response.raise_for_status()

                    wait_time = self.retry_delay * (attempt + 1)

                    print(
                        f"Rate limited (429). "
                        f"Retrying in {wait_time} seconds..."
                    )

                    time.sleep(wait_time)

                    continue

                response.raise_for_status()
                break

            data = response.json()

            records = data.get(data_key, [])
            total = data.get("total", 0)

            all_records.extend(records)

            print(
                f"{endpoint} → "
                f"Fetched {len(records)} records "
                f"(skip={skip})"
            )

            skip += limit

            if skip >= total:
                break

            # Small delay between requests
            time.sleep(0.5)

        return all_records

In [11]:
client = APIClient(
    "https://dummyjson.com",
    max_retries=3,
    retry_delay=2
)

In [12]:
products = client.get_all(
    "/products",
    "products"
)

print("Total products:", len(products))

/products → Fetched 30 records (skip=0)
/products → Fetched 30 records (skip=30)
/products → Fetched 30 records (skip=60)
/products → Fetched 30 records (skip=90)
/products → Fetched 30 records (skip=120)
/products → Fetched 30 records (skip=150)
/products → Fetched 14 records (skip=180)
Total products: 194


In [13]:
products[0]

{'id': 1,
 'title': 'Essence Mascara Lash Princess',
 'description': 'The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.',
 'category': 'beauty',
 'price': 9.99,
 'discountPercentage': 10.48,
 'rating': 2.56,
 'stock': 99,
 'tags': ['beauty', 'mascara'],
 'brand': 'Essence',
 'sku': 'BEA-ESS-ESS-001',
 'weight': 4,
 'dimensions': {'width': 15.14, 'height': 13.08, 'depth': 22.99},
 'warrantyInformation': '1 week warranty',
 'shippingInformation': 'Ships in 3-5 business days',
 'availabilityStatus': 'In Stock',
 'reviews': [{'rating': 3,
   'comment': 'Would not recommend!',
   'date': '2025-04-30T09:41:02.053Z',
   'reviewerName': 'Eleanor Collins',
   'reviewerEmail': 'eleanor.collins@x.dummyjson.com'},
  {'rating': 4,
   'comment': 'Very satisfied!',
   'date': '2025-04-30T09:41:02.053Z',
   'reviewerName': 'Lucas Gordon',
   'reviewerEmail': 'lucas.gordon@x.d

In [14]:
type(products)

list

In [15]:
len(products)

194

In [16]:
import json
from pathlib import Path
from datetime import datetime, timezone


output_dir = Path("data/raw/products")
output_dir.mkdir(parents=True, exist_ok=True)

ingestion_time = datetime.now(timezone.utc)

output = {
    "source": "dummyjson",
    "endpoint": "https://dummyjson.com/products",
    "ingestion_timestamp": ingestion_time.isoformat(),
    "record_count": len(products),
    "products": products
}

file_name = (
    f"products_{ingestion_time.strftime('%Y%m%d_%H%M%S')}.json"
)

file_path = output_dir / file_name

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"Saved to: {file_path}")

Saved to: data\raw\products\products_20260907_205738.json


In [17]:
users = client.get_all(
    "/users",
    "users"
)

print("Total users:", len(users))

/users → Fetched 30 records (skip=0)
/users → Fetched 30 records (skip=30)
/users → Fetched 30 records (skip=60)
/users → Fetched 30 records (skip=90)
Rate limited (429). Retrying in 2 seconds...
/users → Fetched 30 records (skip=120)
/users → Fetched 30 records (skip=150)
Rate limited (429). Retrying in 2 seconds...
/users → Fetched 28 records (skip=180)
Total users: 208


In [22]:
carts = client.get_all(
    "/carts",
    "carts"
)

print("Total carts:", len(carts))

/carts → Fetched 30 records (skip=0)
/carts → Fetched 30 records (skip=30)
/carts → Fetched 30 records (skip=60)
/carts → Fetched 30 records (skip=90)
/carts → Fetched 30 records (skip=120)
/carts → Fetched 30 records (skip=150)
/carts → Fetched 28 records (skip=180)
Total carts: 208


In [23]:
carts[0]

{'id': 1,
 'products': [{'id': 162,
   'title': 'Blue Frock',
   'price': 29.99,
   'quantity': 4,
   'total': 119.96,
   'discountPercentage': 12.13,
   'discountedTotal': 105.41,
   'thumbnail': 'https://cdn.dummyjson.com/product-images/tops/blue-frock/thumbnail.webp'},
  {'id': 113,
   'title': 'Generic Motorcycle',
   'price': 3999.99,
   'quantity': 3,
   'total': 11999.97,
   'discountPercentage': 12.1,
   'discountedTotal': 10547.97,
   'thumbnail': 'https://cdn.dummyjson.com/product-images/motorcycle/generic-motorcycle/thumbnail.webp'},
  {'id': 122,
   'title': 'iPhone 6',
   'price': 299.99,
   'quantity': 3,
   'total': 899.97,
   'discountPercentage': 6.69,
   'discountedTotal': 839.76,
   'thumbnail': 'https://cdn.dummyjson.com/product-images/smartphones/iphone-6/thumbnail.webp'},
  {'id': 138,
   'title': 'Baseball Ball',
   'price': 8.99,
   'quantity': 2,
   'total': 17.98,
   'discountPercentage': 1.71,
   'discountedTotal': 17.67,
   'thumbnail': 'https://cdn.dummyjso

In [24]:
import pandas as pd

carts_df = pd.DataFrame(carts)

carts_df.head()

,id,products,total,discountedTotal,userId,totalProducts,totalQuantity
0,1,"[{'id': 162, 'title': 'Blue Frock', 'price': 2...",13037.88,11510.81,1,4,12
1,2,"[{'id': 86, 'title': 'Man Short Sleeve Shirt',...",139.93,125.70,2,2,7
2,3,"[{'id': 24, 'title': 'Fish Steak', 'price': 14...",1794.85,1625.77,3,6,15
3,4,"[{'id': 92, 'title': 'Sports Sneakers Off Whit...",689.93,636.81,4,2,7
4,5,"[{'id': 161, 'title': 'Samsung Galaxy Tab Whit...",1467.88,1205.80,5,3,12


In [25]:
carts_df.columns

Index(['id', 'products', 'total', 'discountedTotal', 'userId', 'totalProducts',
       'totalQuantity'],
      dtype='object')

In [28]:
cart_items = []

for cart in carts:

    for product in cart["products"]:

        cart_items.append({
            "cart_id": cart["id"],
            "user_id": cart["userId"],
            "product_id": product["id"],
            "quantity": product["quantity"],
            "price": product["price"],
            "total": product["total"],
            "discount_percentage": product.get(
                "discountPercentage"
            )
        })

cart_items_df = pd.DataFrame(cart_items)

cart_items_df.head()

,cart_id,user_id,product_id,quantity,price,total,discount_percentage
0,1,1,162,4,29.99,119.96,12.13
1,1,1,113,3,3999.99,11999.97,12.10
2,1,1,122,3,299.99,899.97,6.69
3,1,1,138,2,8.99,17.98,1.71
4,2,2,86,5,19.99,99.95,6.83


In [29]:
cart_items_df.head()

,cart_id,user_id,product_id,quantity,price,total,discount_percentage
0,1,1,162,4,29.99,119.96,12.13
1,1,1,113,3,3999.99,11999.97,12.10
2,1,1,122,3,299.99,899.97,6.69
3,1,1,138,2,8.99,17.98,1.71
4,2,2,86,5,19.99,99.95,6.83
